# 03 · Split the articles into chunks
Cuts every cleaned article into search-sized pieces (up to 1,000 characters), first at headings and then by size,
adds a context header to each piece, removes duplicates, and saves them to `data/processed/chunks.jsonl`.

In [1]:
# ===== CELL 1: Load the clean docs =====
# Reads the cleaned articles and sets the rules for chunk size.

import re                                                              # re: text patterns
import pandas as pd                                                    # pandas: tables
from langchain_text_splitters import RecursiveCharacterTextSplitter   # A ready-made tool that cuts long text into smaller pieces

docs = pd.read_json("../data/processed/docs_clean.jsonl", lines=True)  # Load the cleaned articles (one JSON record per line) into a table
print("Articles:", len(docs))                                          # How many articles we loaded

CHUNK_SIZE = 1000      # Maximum characters per chunk (about 250 tokens, small enough for the embedding model to read in one go)
CHUNK_OVERLAP = 150    # Characters repeated between neighbouring chunks, so a sentence cut in half still appears whole in one of them
MIN_CHARS = 80         # Pieces shorter than this are too small to be useful, so we drop them

Articles: 324


In [2]:
# ===== CELL 2: Split an article into sections by heading =====
# Cuts an article at its #, ## and ### headings, and remembers the
# heading path (e.g. "Section > Subsection") for every piece.

HEADING = re.compile(r"^(#{1,3})\s+(.+?)\s*#*$")   # Pattern for a heading line: 1 to 3 "#" characters, a space, then the heading text (captured)

def split_sections(text):                          # Define the function; it returns a list of (heading path, section text) pairs
    """Split at #, ## and ### headings. Ignores '#' lines inside code blocks
    (for example, comments in PowerShell or bash samples)."""
    sections = []      # The results go here
    path = {}          # The current heading at each level, e.g. {2: "Implementation", 3: "Built-in filters"}
    lines = []         # The lines of the section we're currently collecting
    in_code = False    # Are we inside a code block right now?

    def flush():                                   # Helper: save the section collected so far
        body = "\n".join(lines).strip()            # Join the collected lines back into one text
        if body:                                   # Only save it if it isn't empty
            heading = " > ".join(path[k] for k in sorted(path) if k > 1)   # Build "Section > Subsection" from levels 2 and 3 (level 1 is the article title, which we already have)
            sections.append((heading, body))       # Add (heading path, text) to the results

    for line in text.split("\n"):                  # Go through the article line by line
        if line.lstrip().startswith("```"):        # A line starting with ``` opens or closes a code block...
            in_code = not in_code                  # ...so flip the in_code switch
        m = None if in_code else HEADING.match(line)   # Only look for headings outside code blocks ("# comment" inside code isn't a heading)
        if m:                                      # If this line is a heading:
            flush()                                #   save the previous section,
            lines = []                             #   start collecting a new one,
            level = len(m.group(1))                #   work out the heading level (the number of "#" characters),
            path = {k: v for k, v in path.items() if k < level}   #   forget headings at this level or deeper (a new ## ends the previous ## and its ###),
            path[level] = m.group(2).strip("` ")   #   and remember this heading's text (without backticks or spaces)
        else:                                      # If it's a normal line:
            lines.append(line)                     #   add it to the current section
    flush()                                        # Save the last section after the loop ends
    return sections                                # Return all the sections

# Test on one article
test = docs[docs["doc_id"] == "blazor/advanced-scenarios"].iloc[0]   # Pick the article we already know well
for heading, body in split_sections(test["text"]):                    # Split it and loop over its sections
    print(f"{len(body):6,} chars | {heading or '(intro)'}")           # Print each section's size and heading ("(intro)" for the text before the first heading)

   395 chars | (intro)
 1,256 chars | Manually build a render tree (`RenderTreeBuilder`)
 1,578 chars | Manually build a render tree (`RenderTreeBuilder`) > Sequence numbers relate to code line numbers and not execution order
 1,510 chars | Manually build a render tree (`RenderTreeBuilder`) > The problem with generating sequence numbers programmatically
 1,477 chars | Manually build a render tree (`RenderTreeBuilder`) > Guidance and conclusions


In [3]:
# ===== CELL 3: Create the chunks =====
# Splits every section into pieces of up to 1,000 characters and stores
# each piece with its article, section, link and a context header.

splitter = RecursiveCharacterTextSplitter(       # Create the text splitter with our settings
    chunk_size=CHUNK_SIZE,                       # Maximum size of each piece
    chunk_overlap=CHUNK_OVERLAP,                 # How much neighbouring pieces share
    separators=["\n\n", "\n", ". ", " ", ""],    # Where to prefer cutting: paragraph break first, then line break, then end of sentence, then space, then anywhere
)

rows = []          # One dictionary per chunk goes here
dropped = 0        # Count of tiny pieces thrown away
for doc in docs.itertuples():                    # Loop over every article (itertuples gives each row with its columns as attributes, e.g. doc.title)
    n = 0                                        # Chunk number within this article, starting at 0
    for heading, body in split_sections(doc.text):     # Split the article into sections
        for piece in splitter.split_text(body):         # Split each section into pieces of up to 1,000 characters
            if len(piece) < MIN_CHARS:                  # If the piece is tiny...
                dropped += 1                            # ...count it...
                continue                                # ...and skip it
            header = doc.title + (f" > {heading}" if heading else "")   # The context header: "Article title > Section > Subsection"
            rows.append({                               # Save the chunk
                "chunk_id": f"{doc.doc_id}::{n}",       # A unique ID, e.g. "blazor/advanced-scenarios::3"
                "doc_id": doc.doc_id,                   # Which article it came from
                "area": doc.area,                       # blazor / fundamentals / mvc
                "title": doc.title,                     # The article title
                "section": heading,                     # The section path inside the article
                "url": doc.url,                         # The link to the article
                "text": piece,                          # The plain passage: shown to the LLM and to the user
                "embed_text": f"{header}\n\n{piece}",   # Used for search: header + blank line + passage
            })
            n += 1                                      # Next chunk number for this article

chunks = pd.DataFrame(rows)                    # Turn all the chunks into a table
print("Chunks created:", len(chunks))          # How many chunks we made
print("Tiny fragments dropped:", dropped)      # How many tiny pieces we skipped

Chunks created: 6886
Tiny fragments dropped: 178


In [4]:
# ===== CELL 4: Remove duplicate chunks =====
# Include files were inserted into several pages, so the same paragraph
# can appear many times. Keep only the first copy of each.

before = len(chunks)                                                    # Number of chunks before removing duplicates
chunks = chunks.drop_duplicates(subset="text", keep="first")           # Remove rows whose "text" is identical to an earlier row
chunks = chunks.reset_index(drop=True)                                  # Renumber the rows 0, 1, 2... after the removal
removed = before - len(chunks)                                          # How many were removed
print(f"Duplicates removed: {removed} ({removed / before:.1%})")       # Print the number, and as a percentage with one decimal place
print("Chunks remaining:", len(chunks))                                 # How many are left

Duplicates removed: 285 (4.1%)
Chunks remaining: 6601


In [5]:
# ===== CELL 5: Check the chunk sizes and save =====
# Shows size statistics and the number of chunks per area, then saves the chunks.

chunks["chars"] = chunks["text"].str.len()              # Add a column with each chunk's length in characters
print(chunks["chars"].describe().round(0))              # Summary statistics: count, average, spread, smallest, quartiles, largest
print()                                                 # An empty line
print(chunks.groupby("area").size().rename("chunks"))   # The number of chunks in each area

chunks.to_json("../data/processed/chunks.jsonl", orient="records", lines=True, force_ascii=False)   # Save all the chunks, one per line, as JSONL
print("\nSaved!")                                       # Confirm it worked

count    6601.0
mean      647.0
std       281.0
min        80.0
25%       400.0
50%       719.0
75%       905.0
max      1000.0
Name: chars, dtype: float64

area
blazor          3568
fundamentals    2270
mvc              763
Name: chunks, dtype: int64

Saved!


In [6]:
# ===== CELL 6: Look at two random chunks =====
# A quick visual check that the chunks read sensibly.

for _, c in chunks.sample(2, random_state=7).iterrows():   # Pick 2 random chunks (random_state=7 means the same 2 every time). iterrows gives (row number, row); "_" means we ignore the number
    print(c["chunk_id"])                 # Its ID
    print("-" * 80)                      # A separator line
    print(c["embed_text"][:800])         # The first 800 characters of its search text (header included)
    print("=" * 80, "\n")                # A separator line and a blank line

fundamentals/portable-object-localization::6
--------------------------------------------------------------------------------
Configure portable object localization in ASP.NET Core > Configuring PO file support in ASP.NET Core > Creating a PO file

Create a file named *\<culture code>.po* in your application root folder. In this example, the file name is *fr.po* because the French language is used:

[Code sample: fr.po]

This file stores both the string to translate and the French-translated string. Translations revert to their parent culture, if necessary. In this example, the *fr.po* file is used if the requested culture is `fr-FR` or `fr-CA`.

mvc/controllers/filters::8
--------------------------------------------------------------------------------
Filters in ASP.NET Core > Implementation > Built-in filter attributes

Responses from the `Multiple` action include the following headers:

* `filter-header: Filter Value`
* `another-filter-header: Another Filter Value`

Several of the f